In [ ]:
!pip install datasets transformers -U

from datasets import load_dataset
from transformers import AutoTokenizer

# Load the Python dataset
ds = load_dataset("Ananda100/python-cleanss", split="train")

# Create a train/validation split (99% train, 1% validation)
ds = ds.train_test_split(test_size=0.01, seed=42)

# Load DeepSeek Coder Tokenizer
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)
vocab_size = len(tokenizer)  # use len(tokenizer), not tokenizer.vocab_size,
                              # since DeepSeek Coder adds special tokens on top of
                              # the base 32000 SentencePiece vocab (real size ~32256)
print(f"DeepSeek Tokenizer Vocab Size: {vocab_size}")

if tokenizer.eos_token_id is None:
    raise ValueError("Tokenizer has no eos_token_id set - fix before tokenizing the dataset.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 135.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1


README.md:   0%|          | 0.00/289 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

data/train-00000-of-00019.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

...

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

DeepSeek Tokenizer Vocab Size: 32022


In [ ]:
import os
import numpy as np
from tqdm.auto import tqdm

def process(example):
    col = 'code' if 'code' in example else 'text' if 'text' in example else 'content'
    ids = tokenizer.encode(example[col], add_special_tokens=False) + [tokenizer.eos_token_id]
    out = {'ids': ids, 'len': len(ids)}
    return out

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=ds['train'].column_names,
        desc="tokenizing the splits",
        num_proc=8,
    )

    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'
        dtype = np.uint16
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()


tokenizing the splits (num_proc=8):   0%|          | 0/1782000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (18530 > 16384). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (16432 > 16384). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (17229 > 16384). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (17430 > 16384). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (16629 > 16384). Running this sequence through the model will result in indexing er

tokenizing the splits (num_proc=8):   0%|          | 0/18000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (16818 > 16384). Running this sequence through the model will result in indexing errors


writing train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

writing test.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

In [ ]:
import numpy as np

train_data = np.memmap('train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('test.bin', dtype=np.uint16, mode='r')

print(f"Total training tokens: {len(train_data):,}")
print(f"Total validation tokens: {len(val_data):,}")


Total training tokens: 2,955,981,351
Total validation tokens: 30,265,069


In [ ]:
import torch
from contextlib import nullcontext

# --- Training / model hyperparameters, defined once, used everywhere below ---
batch_size = 16
block_size = 512
gradient_accumulation_steps = 16
tokens_per_step = batch_size * block_size * gradient_accumulation_steps  # 131,072

# Use the ACTUAL tokenized dataset size (train_data from cell 2) instead of a fixed target.
# num_epochs lets you control how many passes over the real data you want.
num_epochs = 1
target_tokens = len(train_data) * num_epochs
max_iters = int(target_tokens / tokens_per_step)
total_tokens_trained = tokens_per_step * max_iters

learning_rate = 3e-4
warmup_steps = 1000
min_lr = 3e-5
eval_iters = 100

device = "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

# NOTE: removed torch.set_default_device(device) - it silently changes the
# device of every new tensor created anywhere below and can hide real bugs.
# We move things to `device` explicitly instead.
torch.manual_seed(42)

print(f"Targeting Training Tokens: {total_tokens_trained:,}")
print(f"Total optimizer Steps: {max_iters:,}")


Targeting Training Tokens: 2,955,935,744
Total optimizer Steps: 22,552


In [ ]:
def get_batch(split):
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('test.bin', dtype=np.uint16, mode='r')

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    if device_type == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 32256  # overridden below with the real tokenizer vocab size
    n_layer: int = 10        # ~100M params
    n_head: int = 12
    n_embd: int = 768        # 10 layers, 768 embd (12 heads x 64 dim) = ~96M params
    dropout: float = 0.1
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        dev = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"sequence length {t} exceeds block_size {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=dev)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            # Stop early once EOS is generated (only works cleanly for batch size 1)
            if eos_token_id is not None and idx.size(0) == 1 and idx_next.item() == eos_token_id:
                break
        return idx

# Build the model AFTER block_size/vocab_size are known, so config always matches training setup
config = GPTConfig(block_size=block_size, vocab_size=vocab_size)
model = GPT(config)
model = model.to(device)
print(f"Total Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")


Total Parameters: 95.87 M


In [ ]:
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out


In [ ]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

# Model is already on `device` (moved in the model-definition cell above) before
# the optimizer is built, so optimizer state and parameters live on the same device
# from the start.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9)
scheduler_warmup = LinearLR(optimizer, total_iters=warmup_steps)
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max_iters - warmup_steps, eta_min=min_lr)
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps])
scaler = torch.amp.GradScaler(device_type, enabled=(dtype == 'float16'))


In [ ]:
best_val_loss = float('inf')
best_model_params_path = "best_coder_model_100m.pt"
train_loss_list, validation_loss_list = [], []

model.train()

total_loop_iters = max_iters * gradient_accumulation_steps  # actual micro-steps needed

for step in tqdm(range(total_loop_iters)):

    if step % (eval_iters * gradient_accumulation_steps) == 0 and step != 0:
        model.eval()
        losses = estimate_loss(model)
        model.train()

        optimizer_step = step // gradient_accumulation_steps
        print(f"Step {optimizer_step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        train_loss_list.append(losses['train'])
        validation_loss_list.append(losses['val'])

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), best_model_params_path)

    X, y = get_batch("train")

    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((step + 1) % gradient_accumulation_steps == 0) or (step + 1 == total_loop_iters):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

print(f"--- Training Complete ---")
print(f"Total tokens processed: {total_tokens_trained:,}")


  0%|          | 0/360832 [00:00<?, ?it/s]

Step 100: train loss 4.9712, val loss 4.9702
Step 200: train loss 4.3719, val loss 4.3498
Step 300: train loss 3.8422, val loss 3.8215
Step 400: train loss 3.4760, val loss 3.4557
Step 500: train loss 3.0633, val loss 3.0489
Step 600: train loss 2.7158, val loss 2.6612
Step 700: train loss 2.4665, val loss 2.4324
Step 800: train loss 2.2949, val loss 2.3043
Step 900: train loss 2.2252, val loss 2.2338
Step 1000: train loss 2.1370, val loss 2.1261
Step 1100: train loss 2.0656, val loss 2.0887
Step 1200: train loss 2.0299, val loss 2.0065
Step 1300: train loss 1.9852, val loss 1.9666
Step 1400: train loss 1.9320, val loss 1.9473
Step 1500: train loss 1.9041, val loss 1.9132
Step 1600: train loss 1.8882, val loss 1.8775
Step 1700: train loss 1.8617, val loss 1.8641
Step 1800: train loss 1.8516, val loss 1.8406
Step 1900: train loss 1.8417, val loss 1.8138
Step 2000: train loss 1.7805, val loss 1.8185
Step 2100: train loss 1.8042, val loss 1.7760
Step 2200: train loss 1.7867, val loss 1.77

In [ ]:
import torch
from transformers import AutoTokenizer

# --- Load tokenizer (SAME one used for training) ---
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)
vocab_size = len(tokenizer)

# --- Rebuild the exact same model architecture (paste the model-definition cell above this if in a fresh session) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
config = GPTConfig(block_size=512, vocab_size=vocab_size)
model = GPT(config)

# --- Load your trained weights ---
model.load_state_dict(torch.load("best_coder_model_100m.pt", map_location=device))
model.to(device)
model.eval()

def generate_code(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    idx = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        out = model.generate(
            idx,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(out[0].tolist())

prompts = [
    "import asyncio:",
    "import pandas as pd\n\ndef load_data(path):",
    "class BankAccount:\n    def __init__(self, balance):",
]

for p in prompts:
    print("PROMPT:", p)
    print("-" * 60)
    print(generate_code(p))
    print("\n" + "=" * 60 + "\n")


PROMPT: import asyncio:
------------------------------------------------------------
import asyncio:
    import async_timeout
    from functools import partial
except ImportError:  # pragma: no cover
    try:  # pragma: no cover
        import asyncio  # noqa
        from asyncio import coroutine  # noqa
    except ImportError:  # pragma: no cover
        def partial(func):  # pragma: no cover
            if not hasattr(func, '__await__'):
                raise TypeError("asyncio can't be called with a partial API")
            else:
                return func
    from asyncio import EventLoop
    # we don't have to import asyncio here, as a context manager,
    # which will make a future-based coroutine, but not actually
    # the original API.
    from asyncio import Event, EventInWaitingError
    # we only need


PROMPT: import pandas as pd

def load_data(path):
------------------------------------------------------------
import pandas as pd

def load_data(path):
    """loads data 

In [ ]:
# =========================================================
# Push the trained model to Hugging Face Hub (safetensors)
# =========================================================
HF_TOKEN = ""     # <-- your NEWLY generated token
HF_REPO_ID = ""       # <-- pick your actual model name

!pip install -q huggingface_hub safetensors

import json
import torch
from dataclasses import asdict
from safetensors.torch import save_file
from huggingface_hub import HfApi, create_repo

# --- 1. Load your best checkpoint (skip if `model` is already trained/loaded in memory) ---
# model.load_state_dict(torch.load(best_model_params_path, map_location=device))
# model.eval()

# --- 2. Convert the state dict to safetensors ---
state_dict = {k: v.contiguous().clone() for k, v in model.state_dict().items()}
save_file(state_dict, "model.safetensors")

# --- 3. Save the architecture config ---
model_config = asdict(config)
with open("config.json", "w") as f:
    json.dump(model_config, f, indent=2)

# --- 4. Save the tokenizer ---
tokenizer.save_pretrained("hf_upload_tmp")

# --- 5. Write a minimal model card ---
readme = f"""---
license: apache-2.0
tags:
  - nanoGPT
  - code-generation
---

# {HF_REPO_ID.split('/')[-1]}

A GPT-style (nanoGPT architecture) language model trained from scratch on Python code,
using the `deepseek-ai/deepseek-coder-6.7b-base` tokenizer.

- Total tokens trained: {total_tokens_trained:,}
- Block size: {config.block_size}
- Vocab size: {config.vocab_size}

## Loading

This is a **custom architecture**, not a native `transformers` model, so `AutoModel.from_pretrained`
won't work out of the box. To reload:

```python
import json, torch
from safetensors.torch import load_file

with open("config.json") as f:
    cfg = json.load(f)

config = GPTConfig(**cfg)
model = GPT(config)
state_dict = load_file("model.safetensors")
model.load_state_dict(state_dict)
model.eval()
```
"""
with open("hf_upload_tmp/README.md", "w") as f:
    f.write(readme)

import shutil
shutil.move("model.safetensors", "hf_upload_tmp/model.safetensors")
shutil.move("config.json", "hf_upload_tmp/config.json")

# --- 6. Create the repo and upload ---
api = HfApi(token=HF_TOKEN)
create_repo(repo_id=HF_REPO_ID, token=HF_TOKEN, exist_ok=True)

api.upload_folder(
    folder_path="hf_upload_tmp",
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
)

print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}")

Uploaded to https://huggingface.co/Ananda100/pocketcoder100M
